In [1]:
%run ../utility/common_functions.ipynb

In [2]:
session = get_active_session()
session.sql("USE DATABASE CUSTOMER_BILLING_DB").collect()
session.sql("USE SCHEMA GOLD").collect()

run_start_time = datetime.now()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

pipeline_name = "Customer Billing Pipeline"
task_name = "Silver To Gold Layer Transformation"

In [3]:

# Reading YAML files from the Gold stage
from snowflake.snowpark.functions import col, lit, current_timestamp
import yaml
import traceback

gold_stage_path = (
    "@CUSTOMER_BILLING_DB.PUBLIC.AZURE_SNOW_STAGE/"
    "config/yaml/gold/"
)

all_files = session.sql(f"LIST {gold_stage_path}").collect()

yaml_files = [
    row["name"].rsplit("/", 1)[-1]
    for row in all_files
    if row["name"].lower().endswith((".yaml", ".yml"))
]

if not yaml_files:
    raise FileNotFoundError(
        f"No YAML files found in {gold_stage_path}"
    )

print(f"Gold YAML files discovered: {len(yaml_files)}")


# Convert YAML column names to the uppercase Snowflake table columns.
def C(column_name):
    escaped_name = (
        column_name.strip('"').upper().replace('"', '""')
    )
    return col(f'"{escaped_name}"')


for yaml_file_name in yaml_files:

    table_name = yaml_file_name.rsplit(".", 1)[0]
    inserted_count = 0
    updated_count = 0

    run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

    try:
        # Read and parse Gold YAML
        yaml_file_path = gold_stage_path + yaml_file_name
        yaml_content = open_file_read_data(yaml_file_path)
        config = yaml.safe_load(yaml_content)["table"]

        table_name = config["name"]
        target_schema = config["schema_name"]
        source_schema = config["source_schema_name"]
        source_table_name = config.get(
            "source_table_name", table_name
        )

        source_table = f"{source_schema}.{source_table_name}"
        target_table = f"{target_schema}.{table_name}"

        columns = config["columns"]

        column_names = [
            c["name"].strip('"').upper() for c in columns
        ]

        column_types = {
            c["name"].strip('"').upper(): c["data_type"].upper()
            for c in columns
        }

        business_keys = [
            key.strip('"').upper()
            for key in config["business_key"]
        ]

        hash_config = config["hash_key"]
        hash_name = hash_config.get("name", "HASH_KEY").strip('"').upper()

        excluded_columns = [
            c.strip('"').upper()
            for c in config["excluded_columns"]
        ]

        scd = config["scd"]

        if int(scd["type"]) != 1:
            raise ValueError(
                f"{table_name}: YAML must have scd.type = 1"
            )

        inserted_col = scd["inserted_date_column"].strip('"').upper()
        updated_col = scd["updated_date_column"].strip('"').upper()

        metadata_columns = [
            hash_name,
            inserted_col,
            updated_col
        ]

        # Data columns come from the Silver table.
        data_columns = [
            c for c in column_names
            if c not in metadata_columns
        ]

        hash_columns = [
            c for c in data_columns
            if c not in excluded_columns
        ]

        process_log(
            "Start",
            table_name=table_name,
            layer="gold"
        )

        # --------------------------------------------------
        # 1. Read current records from Silver (SCD Type 2)
        # --------------------------------------------------

        silver = (
            session.table(source_table)
            .filter(C("IS_LATEST") == 1)
            .select(
                *[
                    C(c).alias(c)
                    for c in data_columns
                ]
            )
        )

        # Cast Silver columns based on Gold YAML definitions.
        for c in data_columns:
            silver = silver.with_column(
                c,
                C(c).cast(column_types[c])
            )

        validate_business_keys(
            silver,
            business_keys,
            source_table
        )

        # Generate runtime hash key for Silver.
        silver = generate_hash_key(
            silver,
            hash_columns,
            hash_name
        )

        # --------------------------------------------------
        # 2. Read existing Gold records and generate hash
        # --------------------------------------------------

        gold = (
            session.table(target_table)
            .select(
                *[
                    C(c).alias(c)
                    for c in data_columns
                ]
            )
        )

        validate_business_keys(
            gold,
            business_keys,
            target_table
        )

        gold = generate_hash_key(
            gold,
            hash_columns,
            hash_name
        )

        # --------------------------------------------------
        # 3. Identify new and changed records
        # --------------------------------------------------

        s = silver.alias("S")
        g = gold.with_column(
            "_GOLD_EXISTS", lit(1)
        ).alias("G")

        join_condition = None

        for key in business_keys:
            condition = s[key] == g[key]

            join_condition = (
                condition
                if join_condition is None
                else join_condition & condition
            )

        comparison = s.join(
            g,
            join_condition,
            "left"
        )

        is_new = g["_GOLD_EXISTS"].is_null()

        is_changed = (
            g["_GOLD_EXISTS"].is_not_null()
            & (s[hash_name] != g[hash_name])
        )

        records_to_merge = (
            comparison
            .filter(is_new | is_changed)
            .select(
                *[
                    s[c].alias(c)
                    for c in silver.columns
                ]
            )
        )

        # Remove the runtime hash key and create audit columns.
        silver_for_merge = (
            records_to_merge
            .drop(hash_name)
            .with_column(
                inserted_col,
                current_timestamp()
            )
            .with_column(
                updated_col,
                current_timestamp()
            )
            .select(*column_names)
        )

        records_to_merge_count = silver_for_merge.count()

        print(
            f"Records identified for MERGE: "
            f"{records_to_merge_count}"
        )

        # --------------------------------------------------
        # 4. Snowflake MERGE into Gold
        # --------------------------------------------------

        if records_to_merge_count > 0:

            target = session.table(target_table).alias("T")
            source = silver_for_merge.alias("S")

            merge_condition = None

            for key in business_keys:
                condition = target[key] == source[key]

                merge_condition = (
                    condition
                    if merge_condition is None
                    else merge_condition & condition
                )

            # Update non-key data columns and updated timestamp.
            update_values = {
                c: source[c]
                for c in data_columns
                if c not in business_keys
            }

            update_values[updated_col] = current_timestamp()

            # Insert all configured Gold columns.
            insert_values = {
                c: source[c]
                for c in column_names
            }

            merge_result = target.merge(
                source,
                merge_condition,
                [
                    when_matched().update(update_values),
                    when_not_matched().insert(insert_values)
                ]
            )
            
            inserted_count, updated_count = get_delta_merge_counts(
                merge_result
            )
            
            print("Snowflake MERGE completed.")

        # If no new or changed records, no MERGE is required.
        process_log(
            "Success",
            table_name=table_name,
            layer="gold",
            total_inserted_count=inserted_count,
            total_updated_count=updated_count
        )

        print(
            f"Completed {target_table}: "
            f"inserted={inserted_count}, "
            f"updated={updated_count}"
        )

        # Email notification will be handled in the email notebook.

    except Exception as e:

        error_text = traceback.format_exc()

        print(f"Failed {table_name}: {error_text}")

        try:
            process_log(
                "Failed",
                table_name=table_name,
                layer="gold",
                total_inserted_count=inserted_count,
                total_updated_count=updated_count,
                error_message=str(e)
            )

        except Exception as log_error:
            print(
                f"Could not write failure log: {log_error}"
            )

        raise
